# Langfuse Demo — Chapter 4: Evaluation at Scale

---

## The problem: how do I know if my RAG is good?

So far we've seen how to **observe** individual calls.  
But in the real world, you need to answer:

> *"What is the average quality of my assistant across 100 questions?  
> Did my prompt change improve or worsen edge cases?  
> How much did it cost in tokens?"*

## The solution: Datasets + LLM-as-Judge in Batch

```
Dataset "docs-qa-benchmark"
├── Item 1: {question, expected_answer} ──→ RAG pipeline ──→ LLM-judge score
├── Item 2: {question, expected_answer} ──→ RAG pipeline ──→ LLM-judge score
│   ...
└── Item 8: {question, expected_answer} ──→ RAG pipeline ──→ LLM-judge score
                                                                    │
                                                                    ▼
                                                    Dataset Run in the Langfuse UI
                                                    (accuracy, latency, cost)
```

Each "run" is comparable: you can run the same dataset with prompt v1 and v2  
and see which version has higher accuracy in **Dataset Runs** in the UI.

In [1]:
import time
import statistics

from langfuse_demo.client import get_client
from langfuse_demo.config import settings
from langfuse_demo.llm import call_llm
from langfuse_demo.rag import retrieve, rerank, format_context
from langfuse_demo.eval import evaluate_with_llm

from dotenv import load_dotenv

_ = load_dotenv()

langfuse = get_client()
print(f"Langfuse: {settings.langfuse_host}")

Langfuse: http://localhost:3000


---

## 1. Create the Benchmark Dataset

8 question/expected-answer pairs covering all documents in the knowledge base.

In [3]:
DATASET_NAME = "docs-qa-benchmark"

BENCHMARK_ITEMS = [
    {
        "question": "What are the steps to set up the development environment?",
        "expected_answer": "Install Python 3.12, uv, Docker. Clone repo, run uv sync and docker compose up -d. Configure .env.",
        "topic": "onboarding",
    },
    {
        "question": "What should I do when port 5432 is in use?",
        "expected_answer": "Change POSTGRES_PORT in docker-compose.yaml to an available port.",
        "topic": "onboarding",
    },
    {
        "question": "Which databases does the system use?",
        "expected_answer": "PostgreSQL for relational data, ClickHouse for high-volume analytics, Redis for cache and queues.",
        "topic": "arquitetura",
    },
    {
        "question": "What is the system latency and availability SLA?",
        "expected_answer": "P99 latency below 500ms and availability above 99.9%.",
        "topic": "mlops",
    },
    {
        "question": "How does production deployment work?",
        "expected_answer": "Manual approval followed by canary deployment in phases: 5%, 20%, and 100% of traffic.",
        "topic": "mlops",
    },
    {
        "question": "Which LLM models are available in the API?",
        "expected_answer": "llama-3.3-70b-versatile (general use), llama-3.1-8b-instant (low latency), mixtral-8x7b-32768 (long context).",
        "topic": "api_llm",
    },
    {
        "question": "What is a Trace in Langfuse?",
        "expected_answer": "A Trace represents a complete user interaction, containing nested spans and generations.",
        "topic": "langfuse_guia",
    },
    {
        "question": "What are the log data retention rules?",
        "expected_answer": "90 days for training data and 30 days for inference data.",
        "topic": "dados_governanca",
    },
]

print(f"Dataset: '{DATASET_NAME}'")
print(f"Items: {len(BENCHMARK_ITEMS)}")
print()
for i, item in enumerate(BENCHMARK_ITEMS, 1):
    print(f"{i}. [{item['topic']}] {item['question'][:60]}...")

Dataset: 'docs-qa-benchmark'
Items: 8

1. [onboarding] What are the steps to set up the development environment?...
2. [onboarding] What should I do when port 5432 is in use?...
3. [arquitetura] Which databases does the system use?...
4. [mlops] What is the system latency and availability SLA?...
5. [mlops] How does production deployment work?...
6. [api_llm] Which LLM models are available in the API?...
7. [langfuse_guia] What is a Trace in Langfuse?...
8. [dados_governanca] What are the log data retention rules?...


In [4]:
# Create dataset in Langfuse
langfuse.create_dataset(
    name=DATASET_NAME,
    description="Quality benchmark for the internal documentation assistant. 8 questions covering all topics.",
    metadata={"version": "v1", "team": "ds-team"},
)

# Create dataset items
for item in BENCHMARK_ITEMS:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={"question": item["question"]},
        expected_output={"answer": item["expected_answer"]},
        metadata={"topic": item["topic"]},
    )

langfuse.flush()
print(f"Dataset created with {len(BENCHMARK_ITEMS)} items")
print(f"View at: {settings.langfuse_host}/datasets")

Dataset created with 8 items
View at: http://localhost:3000/datasets


---

## 2. Run the RAG Pipeline on Each Item

For each dataset item:
1. Run the RAG pipeline (trace linked to the dataset item)
2. LLM-judge evaluates the answer
3. Score is recorded in Langfuse

In [5]:
RUN_NAME = f"run-rag-v1-{int(time.time())}"

results = []

dataset = langfuse.get_dataset(DATASET_NAME)

print(f"Run: {RUN_NAME}")
print(f"Processing {len(dataset.items)} items...\n")

for i, dataset_item in enumerate(dataset.items, 1):
    question = dataset_item.input["question"]
    expected_answer = dataset_item.expected_output["answer"]

    print(f"[{i}/{len(dataset.items)}] {question[:55]}...")

    t_start = time.time()

    # ── RAG Pipeline ──────────────────────────────────────────────
    with langfuse.start_as_current_observation(
        as_type="span",
        name="rag-benchmark",
        input={"question": question},
        metadata={"run_name": RUN_NAME, "item_topic": dataset_item.metadata.get("topic")},
    ) as pipeline:
        trace_id = langfuse.get_current_trace_id()

        docs = retrieve(question, top_k=2)
        docs = rerank(question, docs)
        context = format_context(docs, max_chars=1000)

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a senior technical assistant for the Data Science team. "
                    "Answer ONLY based on the provided context, clearly and directly. "
                    "If the information is not in the context, say: 'I could not find that information.'"
                ),
            },
            {"role": "user", "content": f"Documentation:\n{context}\n\nQuestion: {question}"},
        ]

        with langfuse.start_as_current_observation(
            as_type="generation",
            name="llm-answer",
            input=messages,
        ) as gen:
            answer, usage = call_llm(messages)
            gen.update(
                model=settings.groq_model,
                output=answer,
                usage_details={"input": usage["input"], "output": usage["output"]},
                cost_details={"input": usage["input_cost"], "output": usage["output_cost"]},
            )

        pipeline.update(output={"answer": answer})

    latency = round(time.time() - t_start, 2)

    # ── Link trace to dataset run (low-level API) ─────────────────
    langfuse.api.dataset_run_items.create(
        run_name=RUN_NAME,
        dataset_item_id=dataset_item.id,
        trace_id=trace_id,
    )

    # ── LLM-judge ─────────────────────────────────────────────────
    score, comment = evaluate_with_llm(question, answer, expected=expected_answer)

    langfuse.create_score(
        trace_id=trace_id,
        name="llm-judge-accuracy",
        value=score,
        data_type="NUMERIC",
        comment=comment,
    )

    results.append({
        "question": question,
        "topic": dataset_item.metadata.get("topic"),
        "score": score,
        "latency_s": latency,
        "input_tokens": usage["input"],
        "output_tokens": usage["output"],
        "input_cost": usage["input_cost"],
        "output_cost": usage["output_cost"],
        "comment": comment,
    })

    print(f"  Score: {score:.2f} | Latency: {latency}s | Tokens: {usage['input']}+{usage['output']}")

langfuse.flush()
print(f"\nRun '{RUN_NAME}' complete!")
print(f"View at: {settings.langfuse_host}/datasets/{DATASET_NAME}/runs")

Run: run-rag-v1-1782813656
Processing 8 items...

[1/8] What are the log data retention rules?...
  Score: 0.90 | Latency: 0.33s | Tokens: 217+24
[2/8] What is a Trace in Langfuse?...
  Score: 0.70 | Latency: 0.18s | Tokens: 255+17
[3/8] Which LLM models are available in the API?...
  Score: 0.80 | Latency: 0.24s | Tokens: 279+52
[4/8] How does production deployment work?...
  Score: 0.90 | Latency: 0.42s | Tokens: 226+44
[5/8] What is the system latency and availability SLA?...
  Score: 0.00 | Latency: 0.14s | Tokens: 254+8
[6/8] Which databases does the system use?...
  Score: 0.95 | Latency: 0.56s | Tokens: 251+43
[7/8] What should I do when port 5432 is in use?...
  Score: 0.90 | Latency: 0.3s | Tokens: 248+24
[8/8] What are the steps to set up the development environmen...
  Score: 0.92 | Latency: 0.3s | Tokens: 246+99

✅ Run 'run-rag-v1-1782813656' complete!
   View at: http://localhost:3000/datasets/docs-qa-benchmark/runs


---

## 3. Results Analysis

In [6]:
scores = [r["score"] for r in results]
latencies = [r["latency_s"] for r in results]
tokens_total = sum(r["input_tokens"] + r["output_tokens"] for r in results)

# Estimated cost (Groq — approximate values)
total_input_tokens = sum(r["input_tokens"] for r in results)
total_output_tokens = sum(r["output_tokens"] for r in results)
estimated_cost = (total_input_tokens * 0.59 + total_output_tokens * 0.79) / 1_000_000

print("=" * 60)
print(f"RESULTS — {RUN_NAME}")
print("=" * 60)
print(f"Items evaluated: {len(results)}")
print(f"Average accuracy: {statistics.mean(scores):.2f}  (0.0–1.0)")
print(f"Median accuracy: {statistics.median(scores):.2f}")
print(f"Min score: {min(scores):.2f}")
print(f"Max score: {max(scores):.2f}")
print(f"Average latency: {statistics.mean(latencies):.2f}s")
print(f"P95 latency: {sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")
print(f"Total tokens: {tokens_total:,}  ({total_input_tokens:,} input + {total_output_tokens:,} output)")
print(f"Estimated cost: ${estimated_cost:.4f} USD")
print("=" * 60)

RESULTS — run-rag-v1-1782813656
Items evaluated: 8
Average accuracy: 0.76  (0.0–1.0)
Median accuracy: 0.90
Min score: 0.00
Max score: 0.95
Average latency: 0.31s
P95 latency: 0.56s
Total tokens: 2,287  (1,976 input + 311 output)
Estimated cost: $0.0014 USD


In [7]:
# Per-item breakdown — identifies where the pipeline failed
print("\nPer-question breakdown (sorted by score):")
print("-" * 60)

for r in sorted(results, key=lambda x: x["score"]):
    emoji = "✅" if r["score"] >= 0.7 else ("⚠️ " if r["score"] >= 0.4 else "❌")
    print(f"{emoji} {r['score']:.2f} [{r['topic']}]")
    print(f"{r['question'][:65]}")
    print(f"→ {r['comment']}")
    print()


Per-question breakdown (sorted by score):
------------------------------------------------------------
❌ 0.00 [mlops]
What is the system latency and availability SLA?
→ response did not provide the required information

✅ 0.70 [langfuse_guia]
What is a Trace in Langfuse?
→ lacking details on nested spans and generations

✅ 0.80 [api_llm]
Which LLM models are available in the API?
→ missing model descriptions

✅ 0.90 [dados_governanca]
What are the log data retention rules?
→ similar to reference, minor formatting difference

✅ 0.90 [mlops]
How does production deployment work?
→ accurate and clear, but slightly more verbose than the reference answer

✅ 0.90 [onboarding]
What should I do when port 5432 is in use?
→ accurate but slightly less concise than reference answer

✅ 0.92 [onboarding]
What are the steps to set up the development environment?
→ accurate and detailed steps to set up development environment

✅ 0.95 [arquitetura]
Which databases does the system use?
→ accurate and cl

---

## What to explore in the UI

1. **Datasets** → `docs-qa-benchmark` → **Runs** tab
   - See the `run-rag-v1-*` run with average score
   - Click any item → see full trace + score side by side

2. **Traces** → filter by tag `benchmark` → see all evaluation traces

3. **Scores** → filter by `llm-judge-accuracy` → score distribution

## Final Summary — What Langfuse solves

| Problem | Langfuse Solution |
|----------|--------------------|
| "Where did the pipeline fail?" | Span hierarchy with inputs/outputs |
| "What is the average quality?" | Datasets + automatic scores |
| "Did my prompt improve?" | Prompt versioning + score per version |
| "How much does it cost?" | `usage_details` → tokens → cost |
| "Who asked what?" | `user_id` + `session_id` |
| "Regression when upgrading model?" | Run same dataset on each deploy |